# Attention maps

See what the trained model looks at. Every `MultiHeadAttentionBlock` in `model.py` saves its last attention table in `self.attention_scores`. This notebook runs one validation sentence through the model and draws those tables as heatmaps: darker means more attention.

You need a trained checkpoint first (`python train.py`). Pick the **ai_env** kernel at the top right.

In [ ]:
import torch
import matplotlib.pyplot as plt

from config import get_config, latest_weights_file_path
from train import get_ds, get_model

In [ ]:
# Load the data, rebuild the model, and load the trained weights into it
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

state = torch.load(latest_weights_file_path(config), map_location=device, weights_only=True)
model.load_state_dict(state["model_state_dict"])
model.eval()    # switch dropout off

## Run one sentence through the model

The real translation goes into the decoder, just like in training, so every row and column of the attention tables lines up with a real token. Run this cell again to get a different sentence.

In [ ]:
batch = next(iter(val_dataloader))      # one random validation sentence
encoder_input = batch["encoder_input"].to(device)
encoder_mask = batch["encoder_mask"].to(device)
decoder_input = batch["decoder_input"].to(device)
decoder_mask = batch["decoder_mask"].to(device)

with torch.no_grad():
    encoder_output = model.encode(encoder_input, encoder_mask)
    model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
# Every attention block now holds the attention table for this sentence.

# token IDs --> token text, for the axis labels
src_tokens = [tokenizer_src.id_to_token(i) for i in encoder_input[0].tolist()]
tgt_tokens = [tokenizer_tgt.id_to_token(i) for i in decoder_input[0].tolist()]
# the real length is the position of the first [PAD]
src_len = src_tokens.index("[PAD]") if "[PAD]" in src_tokens else len(src_tokens)
tgt_len = tgt_tokens.index("[PAD]") if "[PAD]" in tgt_tokens else len(tgt_tokens)

print("Source:", batch["src_text"][0])
print("Target:", batch["tgt_text"][0])

In [ ]:
def get_attention(kind, layer, head):
    """One head's attention table, shape (rows, columns). Each row looks at the columns."""
    if kind == "encoder":       # English looks at English
        block = model.encoder.layers[layer].self_attention_block
    elif kind == "decoder":     # Italian looks at earlier Italian
        block = model.decoder.layers[layer].self_attention_block
    else:                       # "cross": Italian looks at English
        block = model.decoder.layers[layer].cross_attention_block
    return block.attention_scores[0, head].cpu()


def plot_attention(kind, row_tokens, col_tokens, n_rows, n_cols, layers=(0, 1, 2), heads=range(8)):
    """A grid of heatmaps: one row of plots per layer, one column per head."""
    n_rows, n_cols = min(n_rows, 20), min(n_cols, 20)   # cut long sentences at 20 tokens so labels stay readable
    heads = list(heads)
    fig, axes = plt.subplots(len(layers), len(heads), figsize=(2.1 * len(heads), 2.3 * len(layers)), squeeze=False)
    for r, layer in enumerate(layers):
        for c, head in enumerate(heads):
            ax = axes[r][c]
            ax.imshow(get_attention(kind, layer, head)[:n_rows, :n_cols], cmap="Blues", vmin=0)
            ax.set_title(f"layer {layer}, head {head}", fontsize=8)
            # token labels only on the left column and the bottom row
            ax.set_yticks(range(n_rows))
            ax.set_yticklabels(row_tokens[:n_rows] if c == 0 else [], fontsize=6)
            ax.set_xticks(range(n_cols))
            ax.set_xticklabels(col_tokens[:n_cols] if r == len(layers) - 1 else [], rotation=90, fontsize=6)
    fig.suptitle(f"{kind} attention (rows look at columns)")
    fig.tight_layout()
    plt.show()

## Encoder self-attention

Each English word (row) looking at the English words (columns).

In [ ]:
plot_attention("encoder", src_tokens, src_tokens, src_len, src_len)

## Decoder self-attention

Each Italian word looking at the Italian words before it. The empty top-right triangle is the causal mask at work.

In [ ]:
plot_attention("decoder", tgt_tokens, tgt_tokens, tgt_len, tgt_len)

## Cross-attention

Each Italian word (row) looking at the English words (columns). In a well-trained model you can often spot which English word each Italian word is translating.

In [ ]:
plot_attention("cross", tgt_tokens, src_tokens, tgt_len, src_len)